In [26]:
# from silence_tensorflow import silence_tensorflow
# silence_tensorflow()
import numpy as np
import pandas as pd
from tqdm import tqdm
import scipy.stats as ss
from sklearn.metrics import average_precision_score, roc_auc_score
import seaborn as sns
import matplotlib.pyplot as plt
import tensorflow as tf
import elektrum as el

from elektrum import crispr_kinn_predict
from elektrum.crispr_kinn_predict import (
    predict_on_dataframe,
    plot_dataframe,
    get_letter_index,
)
from elektrum.reload import reload_from_dir
from elektrum.encode_seq import get_letter_index, make_alignment, featurize_alignment

from Bio import pairwise2
from Bio.Seq import Seq

import amber

In [27]:
crispr_kinn_predict.config["kinn_1"] = (
    "outputs/2022-05-21/KINN-wtCas9_cleave_rate_log-finkelstein-0-rep4-gRNA1/"
)
# crispr_kinn_predict.config['kinn_2'] = "outputs/2022-05-21/KINN-wtCas9_cleave_rate_log-finkelstein-0-rep5-gRNA2/"
crispr_kinn_predict.manager_kwargs["output_op"] = lambda: tf.keras.layers.Lambda(
    lambda x: tf.math.log(x) / np.log(10), name="output_log"
)

sess = tf.Session()
kinn_1 = reload_from_dir(
    workdir=crispr_kinn_predict.config["kinn_1"],
    sess=sess,
    manager_kwargs=crispr_kinn_predict.manager_kwargs,
    model_fn=crispr_kinn_predict.evo_params["model_fn"],
)

# build rate model
l1 = {l.name: l for l in kinn_1.model.layers}
kinn_1_rate_mod = tf.keras.Model(
    inputs=kinn_1.model.inputs, outputs=l1["gather_rates"].output
)

In [28]:
# Trying to understand featurization of sequences
ltidx = get_letter_index(build_indel=True)
print("Letter index:", ltidx)
print("Letter index shape:", len(ltidx))
ref = Seq("" + "C" * 20 + "G" * 20 + "T" * 20 + "A" * 20 + "GGT")
alt = Seq("A" + "C" * 19 + "G" * 20 + "T" * 20 + "A" * 20 + "GGT")
print("Reference sequence:", ref)
aln = pairwise2.align.localxd(ref, alt, -1, -0.1, -1, 0)
print("Alignment:", aln[0])
# aln = aln[0]
fea = featurize_alignment(aln, ltidx=ltidx, maxlen=len(ref) + 1)
print("Featurized alignment shape:", fea.shape)
print("Featurized change:", np.where(fea == fea.max()))
# fea[np.where(fea == fea.max())]

Letter index: {('A', 'A'): 0, ('C', 'C'): 1, ('G', 'G'): 2, ('T', 'T'): 3, ('A', 'C'): (0, 5), ('A', 'G'): (0, 6), ('A', 'T'): (0, 7), ('C', 'A'): (1, 4), ('C', 'G'): (1, 6), ('C', 'T'): (1, 7), ('G', 'A'): (2, 4), ('G', 'C'): (2, 5), ('G', 'T'): (2, 7), ('T', 'A'): (3, 4), ('T', 'C'): (3, 5), ('T', 'G'): (3, 6), ('-', 'A'): 8, ('-', 'C'): 9, ('-', 'G'): 10, ('-', 'T'): 11, ('_', 'A'): 8, ('_', 'C'): 9, ('_', 'G'): 10, ('_', 'T'): 11, ('A', '-'): (0, 12), ('C', '-'): (1, 12), ('G', '-'): (2, 12), ('T', '-'): (3, 12), ('A', '_'): (0, 12), ('C', '_'): (1, 12), ('G', '_'): (2, 12), ('T', '_'): (3, 12)}
Letter index shape: 32
Reference sequence: CCCCCCCCCCCCCCCCCCCCGGGGGGGGGGGGGGGGGGGGTTTTTTTTTTTTTTTTTTTTAAAAAAAAAAAAAAAAAAAAGGT
Alignment: Alignment(seqA='CCCCCCCCCCCCCCCCCCCCGGGGGGGGGGGGGGGGGGGGTTTTTTTTTTTTTTTTTTTTAAAAAAAAAAAAAAAAAAAAGGT', seqB='ACCCCCCCCCCCCCCCCCCCGGGGGGGGGGGGGGGGGGGGTTTTTTTTTTTTTTTTTTTTAAAAAAAAAAAAAAAAAAAAGGT', score=82.0, start=1, end=83)
Featurized alignment shape: (1, 

In [29]:
print(
    kinn_1.predict(
        fea,
    )
)
print(np.array(kinn_1_rate_mod.predict(kinn_1.blockify_seq_ohe(fea))))


[[-3.4119043]]
[[-3.7081938   3.9214878   0.07545276  0.29897323  1.4328711  -3.427587
   0.24251992]]


# Different rates for different off-target sequences


In [37]:
ltidx = get_letter_index(build_indel=True)
target = "GACGCATAAAGATGAGACGCTGG"[::-1]
print("Target sequence:", target)
ref = Seq(target)

ot = list(target)
ot[-5] = "T"  # Change the last 5th base to T
off_target_end = Seq("".join(ot))

ot = list(target)
ot[-10] = "T"  # Change the last 10th base to T
off_target_mid = Seq("".join(ot))

ot = list(target)
ot[-15] = "T"  # Change the last 15th base to T
off_target_start = Seq("".join(ot))

alignments = []
for ot in tqdm(
    [ref, off_target_end, off_target_mid, off_target_start], desc="Aligning sequences"
):
    aln = pairwise2.align.localxd(target, ot, -1, -0.1, -1, 0)
    alignments.append(aln[0])
alignment_df = pd.DataFrame(
    {"target_seq": [x[0] for x in alignments], "off_seq": [x[1] for x in alignments]}
)
fea = featurize_alignment(alignments, ltidx=ltidx, maxlen=len(ot) + 1)
print("Featurized alignment shape:", fea.shape)


Target sequence: GGTCGCAGAGTAGAAATACGCAG


Aligning sequences: 100%|██████████| 4/4 [00:00<00:00, 1998.95it/s]

Featurized alignment shape: (4, 24, 9)


In [42]:
ltidx = get_letter_index(build_indel=True)
target = "GACGCATAAAGATGAGACGCTGG"[::-1]
print("Target sequence:", target)
ref = Seq(target)


off_target_seqs = []
for i, nt in enumerate(target):
    for new_nt in "ACGT":
        print(f"Changing position {i} from {nt} to {new_nt}")
        if new_nt != nt:
            ot = list(target)
            ot[i] = new_nt
            off_target_seqs.append(Seq("".join(ot)))

# ot = list(target)
# ot[-5] = "T"  # Change the last 5th base to T
# off_target_end = Seq("".join(ot))

# ot = list(target)
# ot[-10] = "T"  # Change the last 10th base to T
# off_target_mid = Seq("".join(ot))

# ot = list(target)
# ot[-15] = "T"  # Change the last 15th base to T
# off_target_start = Seq("".join(ot))

alignments = []
for ot in tqdm([ref]+ off_target_seqs, desc="Aligning sequences"):
    aln = pairwise2.align.localxd(target, ot, -1, -0.1, -1, 0)
    alignments.append(aln[0])
# for ot in tqdm(
#     [ref, off_target_end, off_target_mid, off_target_start], desc="Aligning sequences"
# ):
#     aln = pairwise2.align.localxd(target, ot, -1, -0.1, -1, 0)
#     alignments.append(aln[0])

alignment_df = pd.DataFrame(
    {"target_seq": [x[0] for x in alignments], "off_seq": [x[1] for x in alignments]}
)
fea = featurize_alignment(alignments, ltidx=ltidx, maxlen=len(ot) + 1)
print("Featurized alignment shape:", fea.shape)


Target sequence: GGTCGCAGAGTAGAAATACGCAG
Changing position 0 from G to A
Changing position 0 from G to C
Changing position 0 from G to G
Changing position 0 from G to T
Changing position 1 from G to A
Changing position 1 from G to C
Changing position 1 from G to G
Changing position 1 from G to T
Changing position 2 from T to A
Changing position 2 from T to C
Changing position 2 from T to G
Changing position 2 from T to T
Changing position 3 from C to A
Changing position 3 from C to C
Changing position 3 from C to G
Changing position 3 from C to T
Changing position 4 from G to A
Changing position 4 from G to C
Changing position 4 from G to G
Changing position 4 from G to T
Changing position 5 from C to A
Changing position 5 from C to C
Changing position 5 from C to G
Changing position 5 from C to T
Changing position 6 from A to A
Changing position 6 from A to C
Changing position 6 from A to G
Changing position 6 from A to T
Changing position 7 from G to A
Changing position 7 from G to C

Aligning sequences: 100%|██████████| 70/70 [00:00<00:00, 2225.55it/s]

Featurized alignment shape: (70, 24, 9)


In [43]:
k1_clv_log10 = kinn_1.predict(fea)
k1_log_rates = np.array(kinn_1_rate_mod.predict(kinn_1.blockify_seq_ohe(fea)))
k1_kinn = pd.DataFrame(
    np.hstack([k1_clv_log10, k1_log_rates]),
    columns=[
        "pred_cleavage_log10",
        "k_on_log",
        "k_off_log",
        "k_OI_log",
        "k_IO_log",
        "k_IC_log",
        "k_CI_log",
        "k_cat_log",
    ],
)
full_df = alignment_df.join(k1_kinn)
full_df.to_csv("cas9_competition_all.csv", index=False, sep="\t")


In [34]:
%load_ext watermark
%watermark -n -u -v -iv -w

Last updated: Tue Jul 22 2025

Python implementation: CPython
Python version       : 3.7.3
IPython version      : 7.33.0

elektrum  : 0.1.0
scipy     : 1.5.3
seaborn   : 0.12.2
numpy     : 1.18.5
sklearn   : 1.0.2
amber     : 0.1.3
Bio       : 1.79
tensorflow: 1.14.0
matplotlib: 3.5.1
tqdm      : 4.67.1
pandas    : 1.3.5

Watermark: 2.5.0

